In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A Gaussian processis a non-parameteric tool for regression and function approximation.
It is a distribution function $f(x) \sim \mathcal{GP}(\mu(x), \sigma(x,x^\prime))$.
We start from a prior (in our case mean function $\mu(x)=0$ and covariance function $\sigma(x,x^\prime)$ defining how points are related. 

The folloiwng steps are infolved
1. define mean function
2. define covariance function
3. generate a prior distribution
4. update the prior with new data to get a posterior
5. predict new points using the posterior (aka use the model) 

define a prior on $x \in [0, 30]$

In [ ]:
N = 256
x = np.linspace(0, 30, N)

with mean value $\mu = 0$ and standard deviation $\sigma = 1$.

In [ ]:
mu_prior = np.zeros_like(x)

define correlation matrix

In [ ]:
def squared_exponential_kernel(x1, x2, l=1.0, sigma_f = 1.0):
    """
    x1 ... position 1
    x2 ... position 2
    l ... length scale
    sigma_f ... standard deviaton of mean
    """
    return sigma_f**2 * np.exp(-0.5 * ((x1-x2)/l)**2)

In [ ]:
sigma_f = 1.0 # standard deviation of signal variance
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0, sigma_f)

To get a random sample, that follows the mean and correlation on the domain $\mathcal{D} = [x \in \mathbb{R}; \, 0\leq x \leq 30]$, we use a "multivariate normal sample", which is drawn as follows:

 - define an vector $\vec{x}$ of sample points in $D$
 - compute the covariance matrix $K$ for this vector
 - convert the covariance matrix $K$ into a lower triangular matrix $K = L L^T$ (This is done because $K$ is symmetric and positive definite.)
 - generate a vector $\vec z$ of the same length as $\vec{x}$ that contains independent standard normal samples (mean 0, variance 1)
 - compute $\vec{s} = \mu + L \vec{z}$ to get a sample distribution $\vec{s}$ over $\vec{x}$.  

In [ ]:
# own implementation of: np.random.multivariate_normal (here only for size=1 (number of samples)
# but it easily runs into numeric instabilties 


def multivariate_normal_samples(mean, cov):
    """
    Draw samples from a multivariate normal distribution.

    Parameters:
    - mean: 1D array, mean vector of the distribution.
    - cov: 2D array, covariance matrix of the distribution.

    Returns:
    - sample: 1D array of shape len(mean)
    """
    N = len(mean)
    sample = np.zeros(N)

    # Step 1: Cholesky decomposition of the covariance matrix
    L = np.linalg.cholesky(cov)

    # Step 2: Generate standard normal samples
    z = np.random.normal(size=N)

    # Step 3: Transform to get a sample from N(mean, cov)
    sample = mean + L @ z

    return sample

### Why does this converts uncorrelated noise to exactly the smooth correlated signal (sample) we want? 

The **Cholesky factor** of the correlation matrix $\boldsymbol{K}$ is defined as:
$$ \boldsymbol{K} = \boldsymbol{L} \boldsymbol{L}^T $$
with $\boldsymbol{L}$ being a lower triangular matrix and $\boldsymbol{L}^T$ being its transposed. 

$\vec{z}$ is a vector of  independent (uncorrelated) standard normal samples with mean 0 and variance 1:
$$ z_i \sim \mathcal{N}(0, 1) $$

As the elements of $\vec{z}$ are uncorrelated, the covariance matrix is the identity matrix $\boldsymbol{I}$:
$$ \mathrm{Cov}(\vec{z},\vec{z}) = \boldsymbol{I} $$

The matrix multiplication of the Cholesky factor $\boldsymbol{L}$ with the normal sample $\vec{z}$:
$$ \vec{y} = \boldsymbol{L} \vec{z} $$
is also a vector of same length as $\vec{z}$. 

The covariance matrix of this vector is:
$$ \mathrm{Cov}(\vec{y}, \vec{y}) = \mathrm{Cov}(\boldsymbol{L} \vec{z}, \boldsymbol{L} \vec{z}) = \boldsymbol{L} \mathrm{Cov}(\vec{z}, \vec{z}) \boldsymbol{L}^T = \boldsymbol{L}\boldsymbol{I}\boldsymbol{L}^T = \boldsymbol{L}\boldsymbol{L}^T = \boldsymbol{K} $$

Thus, $\vec{y}$ has the desired covariance matrix.

*Be aware that the covariance matrix is a statistical measure and thus only describes the resulting distribution over many samples.*

In [ ]:
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T

#prior_samples_own = multivariate_normal_samples(mu_prior, K_ss)

In [ ]:
type(None) == type(None)

In [ ]:
def plot_model(title, x, mu, sigma, prior_samples=None, x_train=None, y_train=None, fkt=None):
    
    plt.title(title, fontsize=16)

    plt.fill_between(x, mu-3*sigma, mu+3*sigma, color="gray", alpha=0.1)
    plt.fill_between(x, mu-2*sigma, mu+2*sigma, color="gray", alpha=0.3)
    plt.fill_between(x, mu+sigma, mu+sigma, color="gray", alpha=0.5)
    plt.plot(x, mu, color="gray")

    if type(prior_samples) != type(None):
        plt.plot(x, prior_samples)
    
    if fkt:
        plt.plot(x, data_function(x), "--", color="red", alpha=0.5)
    plt.scatter(x_train, y_train, s=50, color="red")

    plt.xlabel(r"$x$", fontsize=18)
    plt.xticks(fontsize=14)
    plt.xlim(0,30)

    plt.ylabel(r"$y$", fontsize=18)
    plt.yticks(fontsize=14)
    plt.ylim(-4,4)

    plt.tight_layout()
    plt.show()    

In [ ]:
plot_model(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 1.0$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)


## influence of the length scale $l$ used in the covariance matrix 

In [ ]:
sigma_prior = 1.0
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 5)
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T


plot_model(r"$\mu = 0.0$, $l = 5.0$, and $\sigma_f = 1.0$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)



In [ ]:
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 0.2)
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T


plot_model(r"$\mu = 0.0$, $l = 0.2$, and $\sigma_f = 1.0$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)


## influence of non-zero mean function $\mu(x)$

In [ ]:
mu_prior = ((x-15.0)/10.0)**2
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0)
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T

plot_model(r"$\mu = \left(\frac{x-15}{10}\right)^2$, $l = 1.0$, and $\sigma_f = 1.0$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)


## influence of signal variance $\sigma_f^2$

In [ ]:
mu_prior = np.zeros_like(x)
sigma_f = 0.2
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0, sigma_f)
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T

plot_model(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 0.2$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)


In [ ]:
mu_prior = np.zeros_like(x)
sigma_f = 1.3
K_ss = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], 1.0, sigma_f)
prior_samples = np.random.multivariate_normal(mu_prior, K_ss, 3).T

plot_model(r"$\mu = 0.0$, $l = 1.0$, and $\sigma_f = 1.3$",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           None, 
           None,
           None)


# Adding data points / measurements and updating the Gaussian process

In [ ]:
# create data points:
def data_function(x):
    return 2.5 * np.sin(3 * (2*np.pi/30.0) * x)

N_train = 30
sigma_noise_measurement = 0.25
x_train = np.random.uniform(low=0.0, high=30.0, size=N_train)
y_train = data_function(x_train) + np.random.normal(loc=0.0, scale=sigma_noise_measurement, size=N_train)

In [ ]:
# generate prior:

x = np.linspace(0, 30, N)
#x = np.sort(np.concatenate([x, x_train]))

mu_prior = np.zeros_like(x)

l = 1.0
sigma_f = 1.0 # standard deviation of signal variance

K_prior = squared_exponential_kernel(x[:, np.newaxis], x[np.newaxis, :], l, sigma_f)


In [ ]:
# sample prior:

prior_samples = np.random.multivariate_normal(mu_prior, K_prior, 3).T


In [ ]:
# plot prior:

plot_model("prior and data before update",
           x,
           mu_prior,
           sigma_f,
           prior_samples, 
           x_train, 
           y_train,
           data_function)


To train on the new data points, we have to create to more covariance matrices next to the original correlation matrix $\boldsymbol{K}$ (also know as $\boldsymbol{K}_{ss}$:
 - The correlation matrix between the new data points and the prior sample/test points $\boldsymbol{K}_{st}$
 - The correlation matrix between the new data points themself $\boldsymbol{K}_{tt}$



In [ ]:
# compute K_st
K_st = squared_exponential_kernel(x[:, np.newaxis], x_train[np.newaxis, :], l, sigma_f)
print(K_st.shape)

In [ ]:
# compute K_tt
K_tt = squared_exponential_kernel(x_train[:, np.newaxis], x_train[np.newaxis, :], l, sigma_f)
print(K_tt.shape)

In [ ]:
# add measurement noise to K_tt
K_tt += sigma_noise_measurement**2 * np.eye(len(x_train))

In [ ]:
# invert K_tt
K_tt_inv = np.linalg.inv(K_tt)

In [ ]:
# compute posterior mean:
mu_posterior = K_st @ K_tt_inv @ y_train
print(mu_posterior.shape)

In [ ]:
# compute posterior correlation matrix K

K = K_prior - K_st @ K_tt_inv @ K_st.T

print(K.shape)

In [ ]:
var_posterior = np.diag(K)
sigma_posterior = np.sqrt(var_posterior)

In [ ]:
posterior_samples = np.random.multivariate_normal(mu_posterior, K, 3).T


In [ ]:
plot_model("posterior and data",
           x,
           mu_posterior,
           sigma_posterior,
           posterior_samples,#None, 
           x_train, 
           y_train,
           data_function)




## Do this step wise

In [ ]:
# helper function linear interpolation:

def lin_interpol(x, x_data, y_data):
    if x < x_data[0] or x > x_data[-1]:
        raise Exception("Extrapolation not allowed")
        
    idx = np.searchsorted(x_data, x, side='right') - 1
    x0, x1 = x_data[idx], x_data[idx + 1]
    y0, y1 = y_data[idx], y_data[idx + 1]

    return y0 + (y1 - y0) * (x - x0) / (x1 - x0)


In [ ]:
K = K_prior.copy()
mu_posterior = mu_prior.copy()

plot_model("prior and data",
           x,
           mu_posterior,
           np.sqrt(np.diag(K)),
           None, 
           None, 
           None,
           data_function)

for i in range(len(x_train)):
    # compute K_st
    K_st = squared_exponential_kernel(x[:, np.newaxis], x_train[np.newaxis, i], l, sigma_f)
    K_tt = squared_exponential_kernel(x_train[i, np.newaxis], x_train[np.newaxis, i], l, sigma_f) # scalar
    K_tt += sigma_noise_measurement**2
    K_tt_inv = 1.0 / K_tt
    
    mu_prior_at_train = lin_interpol(x_train[i], x, mu_posterior)
    
    mu_posterior += K_st @ (K_tt_inv * (y_train[i] - mu_prior_at_train))
    K = K - K_tt_inv * K_st @ K_st.T
    var_posterior = np.diag(K)
    sigma_posterior = np.sqrt(var_posterior)

    plot_model("posterior and data",
               x,
               mu_posterior,
               sigma_posterior,
               None,#prior_samples, 
               x_train[:i+1], 
               y_train[:i+1],
               data_function)    

In [ ]:
# does not correctly update std/K/correlation yet - need to fix this